## NBA 3.1: one-time load of the inputs and baskets into Unity Catalog (step 3, readiness plan section 10)

Copies the files the 3.1 pipeline reads into managed tables in `kerendia_nba_3_1`, then checks each table against its file:

| Table | Source file (3.1 volume) |
| --- | --- |
| `input_hcp_universe` | `HCP_Universe_without_call_hcps/` (Parquet) |
| `input_claims_cohorts` | `cohorts/claims/cohorts.parquet` (+ `metadata.json` in the table comment) |
| `ref_product_mapping_rx` / `_dx` / `_lab` / `_px`, `ref_dx_flag_additions` | `final_product_mapping/*.csv`, every dated version, column `BASKET_DATE` |

* Values are copied as they are: Parquet columns keep their types; basket values are the exact CSV text (all strings, empty stays empty), with `ROW_NO` (file row order) and `BASKET_DATE` (`YYYY_MM_DD` of the file name).
* Rerunning replaces the tables (same result). Not part of the job. The pipeline only reads these tables when `STORAGE_MODE = "uc"`.
* New basket versions after this load: `save_product_mapping` (00_cohort_config), not a new CSV.

In [0]:
%run ./00_cohort_config

In [0]:
# Sources = the current 3.1 files (volume); targets = the table names used by 00_cohort_config in STORAGE_MODE = "uc".
import csv as _csv_l
import json as _json_l
import os

LOAD_DATE = pd.Timestamp.now().strftime("%Y-%m-%d")
SRC_UNIVERSE = NBA31_HCP_UNIVERSE_PATH
SRC_COHORTS = NBA31_CLAIMS_COHORT_PATH
SRC_COHORT_META = os.path.dirname(NBA31_CLAIMS_COHORT_PATH) + "/metadata.json"
T_UNIVERSE = uc_table("input_hcp_universe")
T_COHORTS = uc_table("input_claims_cohorts")


def set_load_metadata(table, comment, tags):
    """Comment and tags of a loaded table (set_uc_table_metadata only acts in UC mode; this notebook runs in volume mode)."""
    spark.sql(f"COMMENT ON TABLE {table} IS '{_sql_quote(comment)}'")
    if UC_TAGS_ON:
        _tag_sql = ", ".join(f"'{_sql_quote(k)}' = '{_sql_quote(v)}'" for k, v in tags.items())
        spark.sql(f"ALTER TABLE {table} SET TAGS ({_tag_sql})")


def read_basket_text(path):
    """Header and rows of a basket CSV as the exact text of each field (no type guessing, empty stays \"\")."""
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(_csv_l.reader(fh))
    return rows[0], rows[1:]


print(f"Load date {LOAD_DATE} | STORAGE_MODE = {STORAGE_MODE} (files are read, tables are written)")
print(f"Universe: {SRC_UNIVERSE} -> {T_UNIVERSE}\nCohorts:  {SRC_COHORTS} -> {T_COHORTS}\nBaskets:  {PRODUCT_MAPPING_DIR}")

In [0]:
# Copied as it is (all columns and types). The pipeline reads PROVIDER_ID through load_hcp_universe (00_cohort_config).
(spark.read.parquet(SRC_UNIVERSE)
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_UNIVERSE))
set_load_metadata(
    T_UNIVERSE,
    f"NBA 3.1 HCP universe (HCPs without call HCPs): one row per PROVIDER_ID, flags C1_FLAG to C5_FLAG. "
    f"Copied on {LOAD_DATE} from {SRC_UNIVERSE} by 00_load_uc_inputs. Read with load_hcp_universe (00_cohort_config).",
    {"project": "kerendia_nba_3_1", "layer": "input", "dataset": "hcp_universe", "load_date": LOAD_DATE})
print(f"{T_UNIVERSE}: {spark.table(T_UNIVERSE).count():,} rows")

In [0]:
# cohorts.parquet as it is; metadata.json (not read by the code) goes into the table comment.
with open(SRC_COHORT_META) as fh:
    _meta = _json_l.load(fh)
(spark.read.parquet(SRC_COHORTS)
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_COHORTS))
set_load_metadata(
    T_COHORTS,
    f"NBA 3.1 claims cohorts: one row per cohort (cohort_id c1..c28, prediction month, dataset, offset, 12-month "
    f"lookback). Copied on {LOAD_DATE} from {SRC_COHORTS} by 00_load_uc_inputs. Read by 00_cohort_config "
    f"(CLAIMS_COHORT_PATH). metadata.json: {_json_l.dumps(_meta, separators=(',', ':'))}",
    {"project": "kerendia_nba_3_1", "layer": "input", "dataset": "claims_cohorts", "load_date": LOAD_DATE})
print(f"{T_COHORTS}: {spark.table(T_COHORTS).count()} cohorts")

In [0]:
# One table per basket kind with every dated file: the CSV text of each field (all strings), ROW_NO = row order in the
# file, BASKET_DATE = date of the file name. read_mapping_csv (00_cohort_config) rebuilds the file text from it.
BASKET_FILES = {}
for _f in sorted(os.listdir(PRODUCT_MAPPING_DIR)):
    if _f.endswith(".csv"):
        _k, _d = _mapping_kind_date(PRODUCT_MAPPING_DIR + _f)
        BASKET_FILES.setdefault(_k, []).append((_d, PRODUCT_MAPPING_DIR + _f))
assert set(BASKET_FILES) == set(PRODUCT_MAPPING_FILES), f"Basket kinds found {sorted(BASKET_FILES)}"

for kind, versions in sorted(BASKET_FILES.items()):
    header, recs = None, []
    for _d, _p in versions:
        _h, _rows = read_basket_text(_p)
        assert header is None or _h == header, f"{kind}: columns of {_p} differ from the other versions"
        assert all(len(r) == len(_h) for r in _rows), f"{_p}: rows with a different number of fields"
        header = _h
        recs += [tuple(r) + (i, _d) for i, r in enumerate(_rows, 1)]
    _schema = ", ".join(f"`{c}` string" for c in header) + ", ROW_NO int, BASKET_DATE string"
    _table = product_mapping_table(kind)
    (spark.createDataFrame(recs, _schema)
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(_table))
    set_load_metadata(
        _table,
        f"NBA 3.1 {kind} product basket, every version (BASKET_DATE, ROW_NO = file row order; values = CSV text). "
        f"Copied on {LOAD_DATE} from {PRODUCT_MAPPING_DIR} ({', '.join(os.path.basename(p) for _, p in versions)}) "
        f"by 00_load_uc_inputs. Read with product_mapping_path + read_mapping_csv; new versions with save_product_mapping.",
        {"project": "kerendia_nba_3_1", "layer": "ref", "dataset": f"product_mapping_{kind}", "load_date": LOAD_DATE})
    print(f"{_table}: {len(recs):,} rows, versions {[d for d, _ in versions]}")

In [0]:
# Independent of the load cells: files are read again and compared with the tables.
#   Parquet: same columns and types, same row count, no row only in the file or only in the table (exceptAll both ways).
#   Baskets: (a) the field text, row by row; (b) what the notebooks get: pd.read_csv(file, opts) equals
#   read_mapping_csv(file, opts) in UC mode for each option set used in PJ01-PJ05; (c) product_mapping_path picks the
#   same version in UC mode as from the files.
checks = []
for _name, _src, _t in [("input_hcp_universe", SRC_UNIVERSE, T_UNIVERSE), ("input_claims_cohorts", SRC_COHORTS, T_COHORTS)]:
    _fd, _td = spark.read.parquet(_src), spark.table(_t)
    _types = [(f.name, f.dataType.simpleString()) for f in _fd.schema] == [(f.name, f.dataType.simpleString()) for f in _td.schema]
    _diff = _fd.exceptAll(_td).count() + _td.exceptAll(_fd).count()
    _nf, _nt = _fd.count(), _td.count()
    checks.append({"TABLE": _name, "VERSION": "-", "ROWS_FILE": _nf, "ROWS_TABLE": _nt,
                   "OK": _types and _diff == 0 and _nf == _nt,
                   "DETAIL": f"types equal {_types}, rows differing {_diff}"})

_OPTS = [{}, {"dtype": str}, {"dtype": str, "keep_default_na": False}]
_latest_files = {k: product_mapping_path(k) for k in PRODUCT_MAPPING_FILES}   # volume mode (files)
_mode_before = STORAGE_MODE
STORAGE_MODE = "uc"   # read_mapping_csv / product_mapping_path read the tables from here on
try:
    _latest_uc = {k: product_mapping_path(k) for k in PRODUCT_MAPPING_FILES}
    for kind, versions in sorted(BASKET_FILES.items()):
        for _d, _p in versions:
            _h, _rows = read_basket_text(_p)
            _cols, _trows = mapping_rows_uc(kind, _d)
            _text_ok = _cols == _h and [list(r) for r in _trows] == _rows
            _pd_ok = all(pd.read_csv(_p, **o).equals(read_mapping_csv(_p, **o)) for o in _OPTS)
            checks.append({"TABLE": _PRODUCT_MAPPING_TABLES[kind], "VERSION": _d, "ROWS_FILE": len(_rows),
                           "ROWS_TABLE": len(_trows), "OK": _text_ok and _pd_ok,
                           "DETAIL": f"field text equal {_text_ok}, pandas reads equal {_pd_ok}"})
    for k in PRODUCT_MAPPING_FILES:
        checks.append({"TABLE": _PRODUCT_MAPPING_TABLES[k], "VERSION": "latest", "ROWS_FILE": None, "ROWS_TABLE": None,
                       "OK": _latest_uc[k] == _latest_files[k],
                       "DETAIL": f"product_mapping_path: {os.path.basename(_latest_uc[k])}"})
finally:
    STORAGE_MODE = _mode_before

checks = pd.DataFrame(checks)
display(checks)
assert checks.OK.all(), f"Load check failed: {checks[~checks.OK].TABLE.tolist()}"
print(f"All {len(checks)} checks OK: every table equals its file")